# PySpark + Neo4j Graph Demo

This notebook demonstrates an end-to-end pipeline on macOS:

1. **Ingest** CSV files from a mounted drive (`/Volumes/DATA/input`) into PySpark DataFrames.
2. **Load** the data into a Neo4j graph database running in Docker (via the Neo4j Spark connector).
3. **Analyze** the graph with both Cypher (Neo4j) and GraphFrames (in-Spark): PageRank, connected components, triangle count, and top influencers.
4. **Persist** each result set as a Parquet file using `DataFrame.write.parquet(...)`.

Prereqs: Docker is running (`docker compose up -d`), Java 17 is on the PATH, the `.venv` kernel is selected, and `generate_sample_data.py` has populated `/Volumes/DATA/input`.

## 1. Configuration

In [1]:
import os
from pathlib import Path

# --- Paths on the mounted drive ---
INPUT_DIR  = Path(os.environ.get("DEMO_INPUT_DIR",  "/Volumes/DATA/input"))
OUTPUT_DIR = Path(os.environ.get("DEMO_OUTPUT_DIR", "/Volumes/DATA/output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USERS_CSV   = INPUT_DIR / "users.csv"
FRIENDS_CSV = INPUT_DIR / "friendships.csv"

assert USERS_CSV.exists(),   f"Missing {USERS_CSV}. Run: python generate_sample_data.py"
assert FRIENDS_CSV.exists(), f"Missing {FRIENDS_CSV}. Run: python generate_sample_data.py"

# --- Neo4j connection (matches docker-compose.yml) ---
NEO4J_URL      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "cowork-demo-pass"
NEO4J_DATABASE = "neo4j"

print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)
print("Neo4j URL :", NEO4J_URL)

Input dir : /Volumes/DATA/input
Output dir: /Volumes/DATA/output
Neo4j URL : bolt://localhost:7687


## 2. Build the SparkSession

We pull two Maven packages on startup:

- **Neo4j Spark connector** — `org.neo4j:neo4j-connector-apache-spark_2.12:5.3.2_for_spark_3`
- **GraphFrames** — `graphframes:graphframes:0.8.3-spark3.5-s_2.12`

The first run downloads them to `~/.ivy2/`, so it takes ~30 seconds; subsequent runs are instant.

In [2]:
from pyspark.sql import SparkSession

SPARK_PACKAGES = ",".join([
    "org.neo4j:neo4j-connector-apache-spark_2.12:5.3.2_for_spark_3",
    "graphframes:graphframes:0.8.3-spark3.5-s_2.12",
])

spark = (
    SparkSession.builder
        .appName("pyspark-graph-demo")
        .master("local[*]")
        .config("spark.jars.packages", SPARK_PACKAGES)
        .config("spark.jars.repositories", "https://repos.spark-packages.org")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "2g")
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir(str(OUTPUT_DIR / "_checkpoints"))  # required by GraphFrames connectedComponents
print(spark.version)

26/04/23 08:55:16 WARN Utils: Your hostname, Martins-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.232 instead (on interface en0)
26/04/23 08:55:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://repos.spark-packages.org added as a remote repository with the name: repo-1
Ivy Default Cache set to: /Users/martinbenson/.ivy2/cache
The jars for the packages stored in: /Users/martinbenson/.ivy2/jars
org.neo4j#neo4j-connector-apache-spark_2.12 added as a dependency
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c97aba04-2dac-45d5-a197-98a9c5c532fd;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.12;5.3.2_for_spark_3 in central


:: loading settings :: url = jar:file:/Users/martinbenson/claude_code/pyspark_graph_demo/pyspark-graph-demo/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.2_for_spark_3 in central
	found org.neo4j#neo4j-cypher-dsl;2022.11.0 in central
	found org.apiguardian#apiguardian-api;1.1.2 in central
	found org.neo4j.driver#neo4j-java-driver;4.4.18 in central
	found org.reactivestreams#reactive-streams;1.0.4 in central
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in local-m2-cache
:: resolution report :: resolve 251ms :: artifacts dl 11ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.apiguardian#apiguardian-api;1.1.2 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12;5.3.2_for_spark_3 from central in [default]
	org.neo4j#neo4j-connector-apache-spark_2.12_common;5.3.2_for_spark_3 from central in [default]
	org.neo4j#neo4j-cypher-dsl;2022.11.0 from central in [default]
	org.neo4j.driver#neo4j-java-driver;4.4.18 from central in [default]
	org.reactivestr

3.5.1


## 3. Ingest the CSVs from the mounted drive

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

users_schema = StructType([
    StructField("id",   IntegerType(), nullable=False),
    StructField("name", StringType(),  nullable=False),
    StructField("age",  IntegerType(), nullable=True),
    StructField("city", StringType(),  nullable=True),
])

edges_schema = StructType([
    StructField("src_id", IntegerType(), nullable=False),
    StructField("dst_id", IntegerType(), nullable=False),
    StructField("since",  DateType(),    nullable=True),
    StructField("weight", DoubleType(),  nullable=True),
])

users_df = (
    spark.read
         .option("header", True)
         .schema(users_schema)
         .csv(str(USERS_CSV))
)

edges_df = (
    spark.read
         .option("header", True)
         .schema(edges_schema)
         .csv(str(FRIENDS_CSV))
)

print(f"users: {users_df.count()} rows")
print(f"edges: {edges_df.count()} rows")
users_df.show(5, truncate=False)
edges_df.show(5, truncate=False)

users: 200 rows
edges: 1200 rows
+---+--------------+---+--------+
|id |name          |age|city    |
+---+--------------+---+--------+
|1  |Parker Nguyen |19 |Chicago |
|2  |Quinn Martinez|26 |New York|
|3  |Peyton Garcia |55 |Denver  |
|4  |Jordan Chen   |23 |Seattle |
|5  |Quinn Chen    |53 |Seattle |
+---+--------------+---+--------+
only showing top 5 rows

+------+------+----------+------+
|src_id|dst_id|since     |weight|
+------+------+----------+------+
|1     |12    |2020-09-11|0.912 |
|1     |37    |2020-11-14|0.471 |
|1     |127   |2020-10-18|0.648 |
|1     |151   |2018-09-24|0.438 |
|2     |10    |2019-08-15|0.424 |
+------+------+----------+------+
only showing top 5 rows



### Quick EDA

In [4]:
from pyspark.sql import functions as F

users_df.groupBy("city").agg(F.count("*").alias("user_count")) \
        .orderBy(F.desc("user_count")) \
        .show(truncate=False)

edges_df.selectExpr("min(weight) AS min_w", "max(weight) AS max_w", "avg(weight) AS avg_w").show()

+-------------+----------+
|city         |user_count|
+-------------+----------+
|Seattle      |31        |
|Atlanta      |23        |
|Portland     |22        |
|Chicago      |21        |
|Denver       |20        |
|San Francisco|18        |
|New York     |18        |
|Boston       |17        |
|Austin       |15        |
|Los Angeles  |15        |
+-------------+----------+

+-----+-----+------------------+
|min_w|max_w|             avg_w|
+-----+-----+------------------+
|  0.1|0.999|0.5478299999999997|
+-----+-----+------------------+



## 4. Load into Neo4j via the Spark connector

`SaveMode.Overwrite` on a `labels` write performs a `MERGE` keyed on `node.keys`, so the cell is safe to re-run.

In [5]:
neo4j_opts = {
    "url":          NEO4J_URL,
    "authentication.type":     "basic",
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
    "database":     NEO4J_DATABASE,
}

# --- Nodes: (:User {id, name, age, city}) ---
(users_df.write
         .format("org.neo4j.spark.DataSource")
         .mode("Overwrite")
         .options(**neo4j_opts)
         .option("labels", ":User")
         .option("node.keys", "id")
         .save())
print("Nodes written.")

Nodes written.


In [6]:
# --- Relationships: (:User)-[:FRIENDS {since, weight}]->(:User) ---
# The connector expects the src/dst key columns to be surfaced explicitly.
rels_df = (
    edges_df.selectExpr(
        "src_id AS `source.id`",
        "dst_id AS `target.id`",
        "since  AS `rel.since`",
        "weight AS `rel.weight`",
    )
)

(rels_df.write
        .format("org.neo4j.spark.DataSource")
        .mode("Overwrite")
        .options(**neo4j_opts)
        .option("relationship",                     "FRIENDS")
        .option("relationship.save.strategy",       "keys")
        .option("relationship.source.labels",       ":User")
        .option("relationship.source.save.mode",    "match")
        .option("relationship.source.node.keys",    "`source.id`:id")
        .option("relationship.target.labels",       ":User")
        .option("relationship.target.save.mode",    "match")
        .option("relationship.target.node.keys",    "`target.id`:id")
        .option("relationship.properties",          "`rel.since`:since,`rel.weight`:weight")
        .save())
print("Relationships written.")

Relationships written.


## 5. Query Neo4j with Cypher from Spark

This proves the round-trip: we read the graph back out of Neo4j via a Cypher query and land the result as a Spark DataFrame.

In [7]:
node_count_query = "MATCH (u:User) RETURN count(u) AS user_count"
edge_count_query = "MATCH ()-[r:FRIENDS]->() RETURN count(r) AS friendship_count"

def cypher_df(query: str):
    return (
        spark.read
             .format("org.neo4j.spark.DataSource")
             .options(**neo4j_opts)
             .option("query", query)
             .load()
    )

cypher_df(node_count_query).show()
cypher_df(edge_count_query).show()



popular_cities_cypher = """
MATCH (u:User)-[:FRIENDS]->(v:User)
RETURN u.city AS city, count(*) AS outgoing_friendships
"""
(cypher_df(popular_cities_cypher)
    .orderBy(F.desc("outgoing_friendships"))
    .limit(5)
    .show(truncate=False))

+----------+
|user_count|
+----------+
|       200|
+----------+

+----------------+
|friendship_count|
+----------------+
|               0|
+----------------+



26/04/23 08:55:57 WARN Neo4jQueryReadStrategy: Top N push-down optimizations with aggregations are not supported for custom queries.
	These aggregations are going to be ignored.
	Please specify the aggregations in the custom query directly


+----+--------------------+
|city|outgoing_friendships|
+----+--------------------+
+----+--------------------+



## 6. In-Spark graph analytics with GraphFrames

GraphFrames lets us run distributed graph algorithms directly against the Spark DataFrames we already have. No round-trip needed — handy for ad-hoc pipelines.

In [12]:
from graphframes import GraphFrame

vertices = users_df.withColumnRenamed("id", "id")  # GraphFrames requires a column literally named 'id'
edges    = edges_df.withColumnRenamed("src_id", "src").withColumnRenamed("dst_id", "dst")

g = GraphFrame(vertices, edges)
print(f"Vertices: {g.vertices.count()}")
print(f"Edges   : {g.edges.count()}")

# Degree distributions
in_deg  = g.inDegrees.withColumnRenamed("inDegree",  "in_degree")
out_deg = g.outDegrees.withColumnRenamed("outDegree", "out_degree")

Vertices: 200
Edges   : 1200


In [13]:
# --- PageRank (15 iterations, reset probability 0.15) ---
pr = g.pageRank(resetProbability=0.15, maxIter=15)

pagerank_df = (
    pr.vertices
      .select("id", "name", "city", F.round("pagerank", 5).alias("pagerank"))
      .orderBy(F.desc("pagerank"))
)
pagerank_df.show(10, truncate=False)

+---+---------------+-----------+--------+
|id |name           |city       |pagerank|
+---+---------------+-----------+--------+
|99 |Morgan Okafor  |Atlanta    |2.14944 |
|119|Jordan Nguyen  |Denver     |2.03797 |
|91 |Emerson Rossi  |Portland   |1.94664 |
|94 |Cameron Kim    |Chicago    |1.8101  |
|110|Peyton Martinez|Austin     |1.78588 |
|129|Peyton Müller  |Austin     |1.76305 |
|78 |Sam Johnson    |Los Angeles|1.73549 |
|139|Parker Singh   |Denver     |1.72285 |
|12 |Jordan Tanaka  |New York   |1.71021 |
|75 |Cameron Patel  |Boston     |1.68362 |
+---+---------------+-----------+--------+
only showing top 10 rows



In [14]:
# --- Connected components (weakly connected) ---
cc = g.connectedComponents()
components_df = (
    cc.groupBy("component")
      .agg(F.count("*").alias("size"))
      .orderBy(F.desc("size"))
)
components_df.show(5, truncate=False)

+---------+----+
|component|size|
+---------+----+
|1        |200 |
+---------+----+



In [15]:
# --- Triangle count (a simple measure of local clustering) ---
triangles_df = (
    g.triangleCount()
     .select("id", "name", "city", F.col("count").alias("triangles"))
     .orderBy(F.desc("triangles"))
)
triangles_df.show(10, truncate=False)

+---+-------------+-----------+---------+
|id |name         |city       |triangles|
+---+-------------+-----------+---------+
|187|Parker Tanaka|Austin     |49       |
|78 |Sam Johnson  |Los Angeles|46       |
|165|Morgan Chen  |Los Angeles|36       |
|32 |Taylor Kim   |Austin     |36       |
|41 |Parker Singh |Boston     |35       |
|18 |Parker Silva |Boston     |34       |
|121|Finley Nguyen|Los Angeles|34       |
|81 |Jordan Garcia|Los Angeles|34       |
|12 |Jordan Tanaka|New York   |31       |
|70 |Riley Müller |Austin     |31       |
+---+-------------+-----------+---------+
only showing top 10 rows



In [16]:
# --- Combine PageRank + degrees into a single "top influencers" table ---
top_influencers_df = (
    pagerank_df
        .join(in_deg,  on="id", how="left")
        .join(out_deg, on="id", how="left")
        .na.fill({"in_degree": 0, "out_degree": 0})
        .orderBy(F.desc("pagerank"))
        .limit(25)
)
top_influencers_df.show(truncate=False)

+---+---------------+-----------+--------+---------+----------+
|id |name           |city       |pagerank|in_degree|out_degree|
+---+---------------+-----------+--------+---------+----------+
|99 |Morgan Okafor  |Atlanta    |2.14944 |9        |3         |
|119|Jordan Nguyen  |Denver     |2.03797 |11       |7         |
|91 |Emerson Rossi  |Portland   |1.94664 |11       |10        |
|94 |Cameron Kim    |Chicago    |1.8101  |10       |6         |
|110|Peyton Martinez|Austin     |1.78588 |8        |9         |
|129|Peyton Müller  |Austin     |1.76305 |11       |5         |
|78 |Sam Johnson    |Los Angeles|1.73549 |13       |8         |
|139|Parker Singh   |Denver     |1.72285 |8        |9         |
|12 |Jordan Tanaka  |New York   |1.71021 |11       |9         |
|75 |Cameron Patel  |Boston     |1.68362 |10       |5         |
|127|Rowan Müller   |Chicago    |1.66142 |11       |4         |
|189|Parker Okafor  |Portland   |1.65262 |13       |6         |
|14 |Parker Silva   |Seattle    |1.64787

## 7. Persist result sets as Parquet

Each result set is written via `DataFrame.write.parquet(...)` to `/Volumes/DATA/output`. Snappy compression is the Parquet default.

In [17]:
def write_parquet(df, name: str) -> str:
    path = str(OUTPUT_DIR / name)
    (df.write
       .mode("overwrite")
       .parquet(path))
    print(f"  ✓ {name:32s} -> {path}")
    return path

print("Writing Parquet outputs:")
pagerank_path        = write_parquet(pagerank_df,        "pagerank.parquet")
components_path      = write_parquet(components_df,      "components.parquet")
triangles_path       = write_parquet(triangles_df,       "triangles.parquet")
top_influencers_path = write_parquet(top_influencers_df, "top_influencers.parquet")

Writing Parquet outputs:
  ✓ pagerank.parquet                 -> /Volumes/DATA/output/pagerank.parquet
  ✓ components.parquet               -> /Volumes/DATA/output/components.parquet
  ✓ triangles.parquet                -> /Volumes/DATA/output/triangles.parquet
  ✓ top_influencers.parquet          -> /Volumes/DATA/output/top_influencers.parquet


### Verify by reading the Parquet back

In [18]:
for label, path in [
    ("pagerank",        pagerank_path),
    ("components",      components_path),
    ("triangles",       triangles_path),
    ("top_influencers", top_influencers_path),
]:
    df = spark.read.parquet(path)
    print(f"\n{label}: {df.count()} rows, schema:")
    df.printSchema()
    df.show(3, truncate=False)


pagerank: 200 rows, schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- pagerank: double (nullable = true)

+---+-------------+--------+--------+
|id |name         |city    |pagerank|
+---+-------------+--------+--------+
|99 |Morgan Okafor|Atlanta |2.14944 |
|119|Jordan Nguyen|Denver  |2.03797 |
|91 |Emerson Rossi|Portland|1.94664 |
+---+-------------+--------+--------+
only showing top 3 rows


components: 1 rows, schema:
root
 |-- component: long (nullable = true)
 |-- size: long (nullable = true)

+---------+----+
|component|size|
+---------+----+
|1        |200 |
+---------+----+


triangles: 200 rows, schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- triangles: long (nullable = true)

+---+-------------+-----------+---------+
|id |name         |city       |triangles|
+---+-------------+-----------+---------+
|187|Parker Tanaka|Austin

## 8. Bonus: Shortest paths between staff and customers

A second dataset — staff, customers, and a cross-network of connections — lets us demonstrate a classic graph-database question: *given a staff member and a customer who aren't directly in contact, what's the shortest chain of relationships connecting them?*

Input files (also under `/Volumes/DATA/input`):

- `staff.csv` — employees (`staff_id, name, role, department, city`)
- `customers.csv` — customer accounts (`customer_id, name, segment, industry, city`)
- `connections.csv` — all cross-network edges in one table (`src_id, src_type, dst_id, dst_type, rel_type, strength`) covering staff↔staff (org network), customer↔customer (referrals), and staff↔customer (direct touchpoints).

Only ~40% of customers have a direct staff touchpoint in the sample, so shortest-path search is where the value comes from.

In [8]:
STAFF_CSV       = INPUT_DIR / "staff.csv"
CUSTOMERS_CSV   = INPUT_DIR / "customers.csv"
CONNECTIONS_CSV = INPUT_DIR / "connections.csv"

for p in (STAFF_CSV, CUSTOMERS_CSV, CONNECTIONS_CSV):
    assert p.exists(), f"Missing {p}. Run: python generate_sample_data.py"

staff_schema = StructType([
    StructField("staff_id",   IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("role",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("city",       StringType(),  True),
])

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name",        StringType(),  False),
    StructField("segment",     StringType(),  True),
    StructField("industry",    StringType(),  True),
    StructField("city",        StringType(),  True),
])

connections_schema = StructType([
    StructField("src_id",   IntegerType(), False),
    StructField("src_type", StringType(),  False),
    StructField("dst_id",   IntegerType(), False),
    StructField("dst_type", StringType(),  False),
    StructField("rel_type", StringType(),  True),
    StructField("strength", DoubleType(),  True),
])

staff_df       = spark.read.option("header", True).schema(staff_schema).csv(str(STAFF_CSV))
customers_df   = spark.read.option("header", True).schema(customers_schema).csv(str(CUSTOMERS_CSV))
connections_df = spark.read.option("header", True).schema(connections_schema).csv(str(CONNECTIONS_CSV))

print(f"staff:       {staff_df.count()} rows")
print(f"customers:   {customers_df.count()} rows")
print(f"connections: {connections_df.count()} rows")
staff_df.show(3, truncate=False)
customers_df.show(3, truncate=False)
connections_df.groupBy("src_type", "dst_type", "rel_type").count().orderBy("src_type", "dst_type").show(truncate=False)

staff:       30 rows
customers:   50 rows
connections: 146 rows
+--------+--------------+-----------+----------+-----------+
|staff_id|name          |role       |department|city       |
+--------+--------------+-----------+----------+-----------+
|1       |Sam Nguyen    |SDR        |Sales     |New York   |
|2       |Cameron Nguyen|Support Rep|Support   |Los Angeles|
|3       |Rowan Müller  |Support Rep|Support   |Los Angeles|
+--------+--------------+-----------+----------+-----------+
only showing top 3 rows

+-----------+------------+----------+-------------+--------+
|customer_id|name        |segment   |industry     |city    |
+-----------+------------+----------+-------------+--------+
|1          |Nguyen Labs |SMB       |Tech         |Atlanta |
|2          |Kowalski Inc|Enterprise|Finance      |New York|
|3          |Dubois Corp |Enterprise|Manufacturing|Austin  |
+-----------+------------+----------+-------------+--------+
only showing top 3 rows

+--------+--------+-------------

### 8a. Load the staff/customer graph into Neo4j

In [9]:
# --- :Staff nodes ---
(staff_df.write
         .format("org.neo4j.spark.DataSource")
         .mode("Overwrite")
         .options(**neo4j_opts)
         .option("labels", ":Staff")
         .option("node.keys", "staff_id")
         .save())

# --- :Customer nodes ---
(customers_df.write
             .format("org.neo4j.spark.DataSource")
             .mode("Overwrite")
             .options(**neo4j_opts)
             .option("labels", ":Customer")
             .option("node.keys", "customer_id")
             .save())

print("Staff and Customer nodes written.")

Staff and Customer nodes written.


In [10]:
# --- :CONNECTED relationships ---
# The connector's relationship writer binds src/dst node types statically, so we do
# one write per (src_type, dst_type) combo. This is still fully parallel inside Spark.

COMBOS = [
    ("Staff",    "staff_id",    "Staff",    "staff_id"),
    ("Staff",    "staff_id",    "Customer", "customer_id"),
    ("Customer", "customer_id", "Customer", "customer_id"),
    ("Customer", "customer_id", "Staff",    "staff_id"),
]

for src_lbl, src_key, dst_lbl, dst_key in COMBOS:
    subset = (
        connections_df
            .filter((F.col("src_type") == src_lbl) & (F.col("dst_type") == dst_lbl))
            .selectExpr(
                f"src_id AS `source.{src_key}`",
                f"dst_id AS `target.{dst_key}`",
                "rel_type AS `rel.rel_type`",
                "strength AS `rel.strength`",
            )
    )
    if subset.rdd.isEmpty():
        continue
    (subset.write
           .format("org.neo4j.spark.DataSource")
           .mode("Overwrite")
           .options(**neo4j_opts)
           .option("relationship",                  "CONNECTED")
           .option("relationship.save.strategy",    "keys")
           .option("relationship.source.labels",    f":{src_lbl}")
           .option("relationship.source.save.mode", "match")
           .option("relationship.source.node.keys", f"`source.{src_key}`:{src_key}")
           .option("relationship.target.labels",    f":{dst_lbl}")
           .option("relationship.target.save.mode", "match")
           .option("relationship.target.node.keys", f"`target.{dst_key}`:{dst_key}")
           .option("relationship.properties",       "`rel.rel_type`:rel_type,`rel.strength`:strength")
           .save())
    print(f"  ✓ wrote {src_lbl} -[:CONNECTED]-> {dst_lbl}")

  ✓ wrote Staff -[:CONNECTED]-> Staff
  ✓ wrote Staff -[:CONNECTED]-> Customer
  ✓ wrote Customer -[:CONNECTED]-> Customer


### 8b. Shortest path in Cypher (graph-DB-native)

For every staff/customer pair with **no direct edge**, find the shortest chain (up to 6 hops) through the combined network. The query treats relationships as undirected for reach — connection existence is what matters, not who initiated it.

In [11]:
shortest_paths_cypher = """
MATCH (s:Staff), (c:Customer)
WHERE NOT (s)-[:CONNECTED]-(c)
MATCH p = shortestPath( (s)-[:CONNECTED*1..6]-(c) )
RETURN s.staff_id      AS staff_id,
       s.name          AS staff_name,
       s.department    AS department,
       c.customer_id   AS customer_id,
       c.name          AS customer_name,
       c.segment       AS segment,
       length(p)       AS hops,
       [n IN nodes(p) | head(labels(n)) + ':' + coalesce(n.name, '?')] AS path_nodes
ORDER BY hops ASC, staff_id, customer_id
"""

shortest_paths_df = cypher_df(shortest_paths_cypher).cache()
print(f"shortest paths found: {shortest_paths_df.count()}")
shortest_paths_df.show(10, truncate=False)

# Hop-length histogram
shortest_paths_df.groupBy("hops").count().orderBy("hops").show()

shortest paths found: 0
+--------+----------+----------+-----------+-------------+-------+----+----------+
|staff_id|staff_name|department|customer_id|customer_name|segment|hops|path_nodes|
+--------+----------+----------+-----------+-------------+-------+----+----------+
+--------+----------+----------+-----------+-------------+-------+----+----------+

+----+-----+
|hops|count|
+----+-----+
+----+-----+



### 8c. Same question with GraphFrames BFS (in-Spark)

To run shortest-path search inside Spark (no round-trip to Neo4j), we build a single GraphFrame whose vertices are the union of staff + customers. IDs are prefixed (`S:1`, `C:42`) so the two namespaces don't collide, and each connection is materialized in both directions so BFS can traverse the graph like an undirected one.

In [12]:
staff_v = staff_df.select(
    F.concat(F.lit("S:"), F.col("staff_id").cast("string")).alias("id"),
    F.col("name"),
    F.lit("Staff").alias("node_type"),
    F.col("department").alias("group"),
)

cust_v = customers_df.select(
    F.concat(F.lit("C:"), F.col("customer_id").cast("string")).alias("id"),
    F.col("name"),
    F.lit("Customer").alias("node_type"),
    F.col("segment").alias("group"),
)

sc_vertices = staff_v.unionByName(cust_v)

def prefix(type_col, id_col):
    return F.concat(F.when(type_col == "Staff", F.lit("S:")).otherwise(F.lit("C:")), id_col.cast("string"))

forward = connections_df.select(
    prefix(F.col("src_type"), F.col("src_id")).alias("src"),
    prefix(F.col("dst_type"), F.col("dst_id")).alias("dst"),
    F.col("rel_type"),
    F.col("strength"),
)
reverse = forward.select(F.col("dst").alias("src"), F.col("src").alias("dst"), "rel_type", "strength")
sc_edges = forward.unionByName(reverse).distinct()

gc = GraphFrame(sc_vertices, sc_edges)
print(f"staff+customer graph: {gc.vertices.count()} vertices, {gc.edges.count()} edges")

NameError: name 'GraphFrame' is not defined

In [ ]:
# Pick one staff member and one customer who are NOT directly connected,
# then BFS for the shortest path between them (up to 6 hops).
sample = shortest_paths_df.orderBy(F.desc("hops")).limit(1).collect()
if sample:
    sid = int(sample[0]["staff_id"])
    cid = int(sample[0]["customer_id"])
    print(f"BFS from S:{sid} ({sample[0]['staff_name']}) to C:{cid} ({sample[0]['customer_name']})")
    bfs_df = gc.bfs(
        fromExpr=f"id = 'S:{sid}'",
        toExpr=f"id = 'C:{cid}'",
        maxPathLength=6,
    )
    bfs_df.show(1, truncate=False)
else:
    print("No indirect staff↔customer pairs in this sample; everyone has a direct touchpoint.")

### 8d. Persist the staff-to-customer shortest paths

In [ ]:
shortest_paths_out = write_parquet(shortest_paths_df, "shortest_paths.parquet")

# Verify by reading it back
reloaded = spark.read.parquet(shortest_paths_out)
print(f"\nreloaded shortest_paths.parquet: {reloaded.count()} rows")
reloaded.printSchema()
reloaded.orderBy("hops", "staff_id", "customer_id").show(5, truncate=False)

## 9. Shutdown

Stop the Spark session. Neo4j keeps running in Docker; stop it with `docker compose down` when you're done.

In [ ]:
spark.stop()
print("Spark stopped. To stop Neo4j:  docker compose down")